# ANSWER KEY (instructors only)

This notebook contains model solutions and the same self-checks as the student version. Do **not** share it with students.

Alternative correct approaches are acceptable; award marks for correct results and sound method.

---

# Pandas DataFrame Assessment

**Course:** Data Analysis with Python  |  **Total marks:** 100  |  **Suggested time:** 2 to 3 hours

## Purpose
This assessment checks that you can confidently use pandas to **create, clean, explore, summarise, combine, visualise and export** data. When you finish, you should be able to take a raw table of sales data and turn it into insights on your own.

## Instructions
1. Work through the notebook **from top to bottom**. Later tasks depend on earlier ones.
2. Each task tells you **what to do, not how**. Write your own code in the empty code cell under each task.
3. Use the **exact variable names** shown in **bold** (for example `df`, `total_revenue`). The self-checks need them.
4. After a task, run the grey **Self-check** cell. A ✅ means your result matches; a ❌ means review the task and try again. Self-checks do not give you the answer.
5. Where a task says *answer in the markdown cell*, write 1 to 3 sentences in your own words.
6. You may use the official pandas documentation. Work individually.
7. Before submitting, use **Kernel > Restart & Run All** to confirm your notebook runs from start to finish without errors.

## Marking scheme

| Part | Topic | Marks |
|---|---|---|
| A | Create and inspect a DataFrame | 10 |
| B | Data cleaning | 20 |
| C | Creating new columns | 10 |
| D | Selecting and filtering | 20 |
| E | Summaries and grouping | 20 |
| F | Working with dates | 5 |
| G | Combining tables | 5 |
| H | Visualisation | 5 |
| I | Saving and reloading | 5 |
| | **Total** | **100** |

Marks are awarded for **correct results, sensible method and readable code** (clear variable names and short comments where helpful).

---
## Setup
Run the next two cells. Do not change them.

In [ ]:
import hashlib
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _fmt(v):
    if isinstance(v, (bool, np.bool_)):
        return str(bool(v))
    if isinstance(v, (float, np.floating)):
        return f"{float(v):.2f}"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    return str(v)


def check(task, get_value, expected):
    """Self-check helper. Prints whether your result matches the expected answer."""
    try:
        got = hashlib.sha256(_fmt(get_value()).encode()).hexdigest()[:8]
    except Exception as e:
        print(f"❌ {task}: could not run the check ({type(e).__name__}). "
              "Did you complete the task and use the required variable names?")
        return
    print(f"✅ {task}: looks correct" if got == expected
          else f"❌ {task}: your result does not match. Re-read the task and try again.")


### The data
The dictionary below holds 50 sales records. Some values are missing and a few rows are duplicated on purpose.

In [ ]:
# Sales data (one entry per column). Run this cell; do NOT edit it.
sales_dict = {
    "order_id": [
        'ORD-2001', 'ORD-2002', 'ORD-2003', 'ORD-2004', 'ORD-2005',
        'ORD-2006', 'ORD-2007', 'ORD-2008', 'ORD-2009', 'ORD-2010',
        'ORD-2011', 'ORD-2012', 'ORD-2013', 'ORD-2014', 'ORD-2015',
        'ORD-2016', 'ORD-2017', 'ORD-2018', 'ORD-2019', 'ORD-2020',
        'ORD-2021', 'ORD-2022', 'ORD-2023', 'ORD-2024', 'ORD-2025',
        'ORD-2026', 'ORD-2027', 'ORD-2028', 'ORD-2029', 'ORD-2030',
        'ORD-2031', 'ORD-2032', 'ORD-2033', 'ORD-2034', 'ORD-2035',
        'ORD-2036', 'ORD-2037', 'ORD-2038', 'ORD-2039', 'ORD-2040',
        'ORD-2041', 'ORD-2042', 'ORD-2043', 'ORD-2044', 'ORD-2045',
        'ORD-2046', 'ORD-2047', 'ORD-2048', 'ORD-2004', 'ORD-2018',
    ],
    "order_date": [
        '2025-01-22', '2025-04-13', '2025-05-22', '2025-05-14', '2025-01-30',
        '2025-03-24', '2025-06-17', '2025-06-03', '2025-06-20', '2025-05-31',
        '2025-05-10', '2025-03-08', '2025-03-08', '2025-02-17', '2025-05-13',
        '2025-04-27', '2025-03-07', '2025-03-09', '2025-06-23', '2025-03-10',
        '2025-05-30', '2025-01-13', '2025-01-30', '2025-03-07', '2025-06-21',
        '2025-06-03', '2025-05-01', '2025-06-25', '2025-05-12', '2025-02-22',
        '2025-02-05', '2025-02-09', '2025-05-08', '2025-03-20', '2025-01-31',
        '2025-04-16', '2025-02-14', '2025-06-25', '2025-06-03', '2025-02-01',
        '2025-02-16', '2025-02-12', '2025-04-18', '2025-03-10', '2025-03-15',
        '2025-03-11', '2025-03-19', '2025-01-23', '2025-05-14', '2025-03-09',
    ],
    "customer": [
        'Prince Danso', 'Kojo Annan', 'Fiifi Ampofo', 'Abena Osei', 'Akosua Frimpong',
        'Fiifi Ampofo', 'Selina Tetteh', 'Esi Quartey', 'Esi Quartey', 'Fiifi Ampofo',
        'Selina Tetteh', 'Fiifi Ampofo', 'Prince Danso', 'Kojo Annan', 'Kwesi Adjei',
        'Selina Tetteh', 'Kojo Annan', 'Kwesi Adjei', 'Esi Quartey', 'Prince Danso',
        np.nan, 'Kojo Annan', 'Kwesi Adjei', 'Esi Quartey', 'Abena Osei',
        'Kwesi Adjei', 'Akosua Frimpong', 'Esi Quartey', 'Kojo Annan', 'Kwesi Adjei',
        'Nana Yeboah', 'Prince Danso', 'Nana Yeboah', 'Kwesi Adjei', 'Prince Danso',
        'Yaa Asantewaa', 'Kojo Annan', 'Mavis Larbi', 'Abena Osei', np.nan,
        'Mavis Larbi', 'Nana Yeboah', 'Yaa Asantewaa', 'Abena Osei', 'Fiifi Ampofo',
        np.nan, 'Prince Danso', 'Akosua Frimpong', 'Abena Osei', 'Kwesi Adjei',
    ],
    "region": [
        'Northern', 'Greater Accra', 'Greater Accra', 'Volta', 'Volta',
        'Northern', 'Greater Accra', 'Ashanti', 'Volta', 'Volta',
        'Northern', 'Greater Accra', 'Ashanti', 'Volta', 'Western',
        'Northern', 'Western', 'Volta', 'Volta', 'Western',
        'Volta', 'Western', 'Western', 'Greater Accra', 'Western',
        'Ashanti', 'Ashanti', 'Western', 'Western', 'Ashanti',
        'Ashanti', 'Greater Accra', 'Ashanti', 'Ashanti', 'Ashanti',
        'Volta', 'Volta', 'Western', 'Western', 'Ashanti',
        'Volta', 'Volta', 'Volta', 'Volta', 'Ashanti',
        'Greater Accra', 'Western', 'Northern', 'Volta', 'Volta',
    ],
    "product": [
        'Power Bank', 'Printer Paper', 'Laptop', 'Notebook Pack', 'Laptop',
        'Headphones', 'Printer Paper', 'Power Bank', 'Headphones', 'Power Bank',
        'Laptop', 'Headphones', 'Notebook Pack', 'Backpack', 'Notebook Pack',
        'Power Bank', 'Headphones', 'Headphones', 'Office Chair', 'Power Bank',
        'Power Bank', 'Printer Paper', 'Desk', 'Notebook Pack', 'Laptop',
        'Notebook Pack', 'Backpack', 'Smartphone', 'Laptop', 'Laptop',
        'Laptop', 'Power Bank', 'Printer Paper', 'Office Chair', 'Headphones',
        'Power Bank', 'Notebook Pack', 'Desk', 'Office Chair', 'Smartphone',
        'Notebook Pack', 'Office Chair', 'Smartphone', 'Notebook Pack', 'Backpack',
        'Laptop', 'Desk', 'Printer Paper', 'Notebook Pack', 'Headphones',
    ],
    "category": [
        'Accessories', 'Stationery', 'Electronics', 'Stationery', 'Electronics',
        'Electronics', 'Stationery', 'Accessories', 'Electronics', 'Accessories',
        'Electronics', 'Electronics', 'Stationery', 'Accessories', 'Stationery',
        'Accessories', 'Electronics', 'Electronics', 'Furniture', 'Accessories',
        'Accessories', 'Stationery', 'Furniture', 'Stationery', 'Electronics',
        'Stationery', 'Accessories', 'Electronics', 'Electronics', 'Electronics',
        'Electronics', 'Accessories', 'Stationery', 'Furniture', 'Electronics',
        'Accessories', 'Stationery', 'Furniture', 'Furniture', 'Electronics',
        'Stationery', 'Furniture', 'Electronics', 'Stationery', 'Accessories',
        'Electronics', 'Furniture', 'Stationery', 'Stationery', 'Electronics',
    ],
    "quantity": [
        3, 2, 4, 2, 6,
        9, 6, 5, 5, 10,
        1, 2, 6, 3, 9,
        10, 4, 1, 5, 4,
        3, 2, 9, 8, 5,
        3, 4, 4, 8, 5,
        8, 2, 6, 10, 1,
        2, 6, 6, 3, 10,
        7, 1, 7, 9, 3,
        4, 6, 6, 2, 1,
    ],
    "unit_price": [
        220.64, 97.54, 6675.29, 60.28, 6231.29,
        463.46, 98.38, 230.99, 452.88, 212.38,
        6308.7, 453.7, 61.45, 272.74, 59.85,
        222.59, 460.75, 465.92, 1163.12, 228.84,
        224.33, 97.96, 1714.95, 57.01, 6601.71,
        57.92, 283.14, 3058.37, 6364.63, 6362.11,
        6669.68, 226.06, 98.55, 1196.19, 464.43,
        217.72, 57.62, 1734.54, 1203.86, 3350.1,
        61.98, 1192.41, 3192.2, 60.45, 289.41,
        6370.05, 1790.47, 95.18, 60.28, 465.92,
    ],
    "discount": [
        0.1, 0.15, 0.0, 0.05, 0.15,
        0.15, 0.05, 0.1, 0.05, 0.15,
        0.05, np.nan, 0.05, 0.15, np.nan,
        0.1, 0.15, 0.0, 0.0, 0.15,
        0.0, 0.15, 0.05, np.nan, 0.0,
        0.05, 0.0, 0.05, 0.0, np.nan,
        0.05, 0.05, 0.05, 0.1, 0.1,
        0.1, 0.0, np.nan, 0.1, 0.1,
        0.15, 0.05, 0.05, 0.1, 0.05,
        0.05, 0.15, 0.0, 0.05, 0.0,
    ],
    "payment_method": [
        'Bank Transfer', 'Cash', 'Mobile Money', np.nan, 'Credit Card',
        'Bank Transfer', 'Credit Card', 'Credit Card', 'Cash', np.nan,
        'Mobile Money', 'Mobile Money', 'Bank Transfer', 'Credit Card', 'Cash',
        'Credit Card', 'Mobile Money', 'Cash', 'Bank Transfer', 'Cash',
        'Bank Transfer', 'Bank Transfer', 'Bank Transfer', 'Credit Card', 'Mobile Money',
        np.nan, 'Bank Transfer', 'Bank Transfer', 'Cash', 'Mobile Money',
        'Cash', 'Credit Card', 'Mobile Money', 'Bank Transfer', 'Mobile Money',
        'Bank Transfer', 'Cash', 'Bank Transfer', 'Credit Card', 'Cash',
        'Bank Transfer', 'Credit Card', 'Credit Card', np.nan, 'Mobile Money',
        'Cash', 'Mobile Money', 'Credit Card', np.nan, 'Cash',
    ],
    "sales_rep": [
        'Ama Serwaa', 'Efua Owusu', 'Efua Owusu', 'Ama Serwaa', 'Efua Owusu',
        'Kwame Mensah', 'Ama Serwaa', 'Kwame Mensah', 'Kofi Boateng', 'Ama Serwaa',
        'Ama Serwaa', 'Ama Serwaa', 'Ama Serwaa', 'Ama Serwaa', 'Ama Serwaa',
        'Kwame Mensah', 'Kwame Mensah', 'Kofi Boateng', 'Efua Owusu', 'Kofi Boateng',
        'Efua Owusu', 'Kofi Boateng', 'Kofi Boateng', 'Kofi Boateng', 'Kofi Boateng',
        'Kwame Mensah', 'Kofi Boateng', 'Ama Serwaa', 'Kofi Boateng', 'Kwame Mensah',
        'Kwame Mensah', 'Ama Serwaa', 'Kofi Boateng', 'Kwame Mensah', 'Ama Serwaa',
        'Ama Serwaa', 'Kofi Boateng', 'Kwame Mensah', 'Kofi Boateng', 'Ama Serwaa',
        'Kwame Mensah', 'Ama Serwaa', 'Kofi Boateng', 'Ama Serwaa', 'Ama Serwaa',
        'Kofi Boateng', 'Efua Owusu', 'Efua Owusu', 'Ama Serwaa', 'Kofi Boateng',
    ],
    "rating": [
        np.nan, 1.0, 4.0, 5.0, 5.0,
        5.0, np.nan, 5.0, 5.0, 3.0,
        4.0, 2.0, 5.0, 1.0, 3.0,
        2.0, 1.0, 3.0, 3.0, 1.0,
        3.0, 1.0, np.nan, 1.0, 5.0,
        np.nan, np.nan, 2.0, 2.0, 2.0,
        2.0, 3.0, np.nan, 2.0, 4.0,
        1.0, 1.0, np.nan, 2.0, 2.0,
        3.0, 1.0, 2.0, np.nan, 3.0,
        1.0, 3.0, 5.0, 5.0, 3.0,
    ],
}

# Sales representative details (used in the merge task)
rep_info = {
    "sales_rep": [
        'Kwame Mensah', 'Ama Serwaa', 'Kofi Boateng', 'Efua Owusu',
    ],
    "team": [
        'North', 'South', 'North', 'South',
    ],
    "years_experience": [
        6, 3, 8, 2,
    ],
}

---

---
## Part A: Create and inspect a DataFrame (10 marks)
**Focus:** turning raw Python data into a DataFrame and getting a first feel for it.

### Task 1: Create the DataFrame (3 marks)

Convert the dictionary `sales_dict` into a pandas DataFrame and store it in a variable named **`df`**.

In [ ]:
df = pd.DataFrame(sales_dict)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 1", lambda: df.shape, "53844601")


### Task 2: View the first 10 records (2 marks)

Display the **first 10 records** of `df`.

In [ ]:
df.head(10)

### Task 3: View the last 5 records (2 marks)

Display the **last 5 records** of `df`.

In [ ]:
df.tail(5)

### Task 4: Summarise the structure (3 marks)

Display a concise summary of `df` showing the number of rows and columns, each column's data type and its count of non-null values.

**Then answer in the markdown cell below:** which two columns do you think have the wrong data type for what they contain, and which columns appear to have missing data?

In [ ]:
df.info()

**Model answer:** `order_date` is stored as text (object) and `rating` and `discount` are floats because of missing values; missing data appears in customer, discount, payment_method and rating (compare non-null counts with the number of rows).

---
## Part B: Data cleaning (20 marks)
**Focus:** finding and fixing problems before analysing. Real data is never clean.

### Task 5: Count missing values (3 marks)

Count the number of missing values in **each column**. Store the result in **`missing_counts`**.

*Hint: Think of one method that flags missing values and another that adds them up.*

In [ ]:
missing_counts = df.isna().sum()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 5", lambda: missing_counts.sort_index().to_dict(), "6d4307d7")


### Task 6: Percentage of missing values (3 marks)

Calculate the **percentage** of missing values in each column, rounded to 1 decimal place. Store it in **`missing_pct`**.

In [ ]:
missing_pct = (df.isna().mean() * 100).round(1)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 6", lambda: missing_pct.sort_index().round(1).to_dict(), "f1f2593f")


### Task 7: Find duplicate rows (3 marks)

Find how many **fully duplicated rows** exist in `df`. Store the number in **`dup_count`**.

In [ ]:
dup_count = df.duplicated().sum()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 7", lambda: int(dup_count), "d4735e3a")


### Task 8: Remove duplicates (2 marks)

Remove the duplicated rows (keep the first occurrence) and **reset the index** so it runs 0, 1, 2, ... again. Update `df`.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 8", lambda: df.shape, "edfc18b9")


### Task 9: Fill text and discount gaps (3 marks)

Handle missing values as follows, updating `df`:
- `discount`: assume **no discount** (0)
- `payment_method` and `customer`: fill with the text **"Unknown"**

*Hint: Only the rating column should still contain missing values after this step.*

In [ ]:
df["discount"] = df["discount"].fillna(0)
df["payment_method"] = df["payment_method"].fillna("Unknown")
df["customer"] = df["customer"].fillna("Unknown")

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 9", lambda: int(df.isna().sum().sum()), "2c624232")


### Task 10: Handle missing ratings (3 marks)

Customers often skip ratings, so the fact that a rating is missing may be useful.

1. First create a new True/False column **`rating_missing`** that records which rows had no rating.
2. Then fill the missing ratings with the **median** rating.

In [ ]:
df["rating_missing"] = df["rating"].isna()
df["rating"] = df["rating"].fillna(df["rating"].median())

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 10", lambda: (int(df["rating_missing"].sum()), int(df["rating"].isna().sum())), "a52f3605")


### Task 11: Fix the date column (3 marks)

The `order_date` column is stored as text. Convert it to a proper **datetime** type.

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 11", lambda: pd.api.types.is_datetime64_any_dtype(df["order_date"]), "3cbc87c7")


---
## Part C: Creating new columns (10 marks)
**Focus:** deriving new information from existing columns.

### Task 12: Calculate total sales (4 marks)

Create a column **`total_sales`** for each order:

`quantity x unit_price x (1 - discount)`, rounded to **2 decimal places**.

In [ ]:
df["total_sales"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 12", lambda: df["total_sales"].sum(), "bee153c7")


### Task 13: Extract the month (2 marks)

Create a column **`month`** containing the **month name** (for example "January") taken from `order_date`.

In [ ]:
df["month"] = df["order_date"].dt.month_name()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 13", lambda: df["month"].nunique(), "e7f6c011")


### Task 14: Classify order size (4 marks)

Create a column **`order_size`** using these rules on `total_sales`:
- **"Large"** if 10,000 or more
- **"Medium"** if 2,000 or more (but below 10,000)
- **"Small"** otherwise

Then store the count of each size in **`order_size_counts`**: a dictionary sorted alphabetically by size.

*Hint: You need a way to apply a condition across a whole column, and you may need to nest or repeat it.*

In [ ]:
df["order_size"] = np.where(df["total_sales"] >= 10000, "Large",
                           np.where(df["total_sales"] >= 2000, "Medium", "Small"))
order_size_counts = df["order_size"].value_counts().sort_index().to_dict()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 14", lambda: order_size_counts, "e3c65638")


---
## Part D: Selecting and filtering (20 marks)
**Focus:** pulling out exactly the columns and rows you need.

### Task 15: Select columns (2 marks)

Select only the columns **`product`**, **`quantity`** and **`total_sales`**. Store them in **`subset`**.

In [ ]:
subset = df[["product", "quantity", "total_sales"]]

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 15", lambda: subset.shape, "d2862785")


### Task 16: Select rows by position and by label (2 marks)

1. Display the **first row** of `df` using its **position**.
2. Display the rows with **index labels 5 to 9** (inclusive) using their **labels**.

**Then explain in the markdown cell below** one difference between selecting by position and by label.

In [ ]:
display(df.iloc[0])
display(df.loc[5:9])

**Model answer:** Position (`iloc`) counts rows from 0 regardless of the index labels; label (`loc`) uses the index labels and includes the end of a slice.

### Task 17: Filter by one condition (3 marks)

Find all orders from the **Ashanti** region. Store them in **`ashanti_orders`**.

In [ ]:
ashanti_orders = df[df["region"] == "Ashanti"]

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 17", lambda: len(ashanti_orders), "4fc82b26")


### Task 18: Filter by two conditions (AND) (3 marks)

Find all **Electronics** orders where the **quantity is 5 or more**. Store them in **`bulk_electronics`**.

In [ ]:
bulk_electronics = df[(df["category"] == "Electronics") & (df["quantity"] >= 5)]

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 18", lambda: len(bulk_electronics), "19581e27")


### Task 19: Filter by a list of values (3 marks)

Find all orders from **Volta or Northern** regions, using a method that checks membership in a list. Store them in **`north_volta`**.

In [ ]:
north_volta = df[df["region"].isin(["Volta", "Northern"])]

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 19", lambda: len(north_volta), "9400f1b2")


### Task 20: Combine different conditions (3 marks)

Find orders that had a **discount greater than 0** and a **rating of 4 or higher**. Store them in **`good_discount`**.

In [ ]:
good_discount = df[(df["discount"] > 0) & (df["rating"] >= 4)]

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 20", lambda: len(good_discount), "2c624232")


### Task 21: Sort and rank (4 marks)

Find the **5 orders with the highest `total_sales`**, highest first. Store them in **`top5`**.

In [ ]:
top5 = df.sort_values("total_sales", ascending=False).head(5)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 21", lambda: top5["order_id"].tolist(), "e91ce79b")


---
## Part E: Summaries and grouping (20 marks)
**Focus:** turning rows into answers a manager can use.

### Task 22: Find the sum of sales (2 marks)

Find the **total of all sales** (`total_sales`). Store it in **`total_revenue`**.

In [ ]:
total_revenue = df["total_sales"].sum()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 22", lambda: total_revenue, "bee153c7")


### Task 23: Basic statistics (3 marks)

Find the **mean, median, maximum and minimum** of `total_sales`. Display them together, and store the mean in **`avg_sales`**.

**Then answer in the markdown cell below:** why is the mean much higher than the median here?

In [ ]:
avg_sales = df["total_sales"].mean()
df["total_sales"].agg(["mean", "median", "max", "min"])

**Model answer:** A few very large orders (such as laptops in big quantities) pull the mean upward, while the median is not affected by extreme values, so the distribution is right-skewed.

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 23", lambda: avg_sales, "f1e5e666")


### Task 24: Sales by region (3 marks)

Calculate **total sales for each region**, sorted from highest to lowest. Store it in **`sales_by_region`**.

In [ ]:
sales_by_region = df.groupby("region")["total_sales"].sum().sort_values(ascending=False)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 24", lambda: sales_by_region.sort_index().round(2).to_dict(), "cc71840a")


### Task 25: Multi-measure summary by category (4 marks)

For each **category**, produce one table with these columns (use exactly these names) and store it in **`category_summary`**:
- `orders`: number of orders
- `revenue`: total sales
- `avg_quantity`: average quantity
- `avg_rating`: average rating

*Hint: Look up named aggregation in groupby.*

In [ ]:
category_summary = df.groupby("category").agg(
    orders=("order_id", "count"),
    revenue=("total_sales", "sum"),
    avg_quantity=("quantity", "mean"),
    avg_rating=("rating", "mean"),
)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 25", lambda: category_summary["revenue"].sort_index().round(2).to_dict(), "462893b8")


### Task 26: Pivot table (4 marks)

Build a pivot table with **regions as rows**, **categories as columns** and the **sum of `total_sales`** as values, including **grand totals**. Store it in **`pivot`**.

In [ ]:
pivot = pd.pivot_table(df, values="total_sales", index="region", columns="category", aggfunc="sum", margins=True)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 26", lambda: pivot.loc["All", "All"], "bee153c7")


### Task 27: Business questions (4 marks)

Answer both questions with code:
1. Which **product** sold the most **units** in total? Store its name in **`best_product`**.
2. Which **customer** spent the most in total, **ignoring "Unknown"**? Store the name in **`top_customer`**.

In [ ]:
best_product = df.groupby("product")["quantity"].sum().idxmax()
top_customer = df[df["customer"] != "Unknown"].groupby("customer")["total_sales"].sum().idxmax()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 27", lambda: (best_product, top_customer), "c5045ce5")


---
## Part F: Working with dates (5 marks)
**Focus:** analysing trends over time.

### Task 28: Monthly sales trend (5 marks)

Calculate **total sales per month**, in **chronological order** (January first, not alphabetical). Store it in **`monthly_sales`**.

*Hint: Grouping by the month name will sort alphabetically. Consider grouping by a period derived from the date.*

In [ ]:
monthly_sales = df.groupby(df["order_date"].dt.to_period("M"))["total_sales"].sum()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 28", lambda: monthly_sales.round(2).tolist(), "f2c04223")


---
## Part G: Combining tables (5 marks)
**Focus:** enriching one table with information from another.

### Task 29: Merge sales with representative details (5 marks)

1. Convert the dictionary `rep_info` into a DataFrame named **`rep_df`**.
2. Add the rep's `team` and `years_experience` to every sale by joining on **`sales_rep`**, keeping **all sales rows**. Store the result in **`df_merged`**.
3. Calculate **total sales per team** and store it in **`team_sales`**.

In [ ]:
rep_df = pd.DataFrame(rep_info)
df_merged = df.merge(rep_df, on="sales_rep", how="left")
team_sales = df_merged.groupby("team")["total_sales"].sum()

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 29", lambda: df_merged.shape, "0fa2ee90")
check("Task 29", lambda: team_sales.sort_index().round(2).to_dict(), "02b2c47a")


---
## Part H: Visualisation (5 marks)
**Focus:** communicating results with clear charts.

### Task 30: Bar chart (3 marks)

Draw a **bar chart of total sales by region**. It must have a **title**, labelled **axes** and be readable (no overlapping labels).

In [ ]:
sales_by_region.plot(kind="bar", color="#1F3A5F")
plt.title("Total sales by region")
plt.xlabel("Region")
plt.ylabel("Sales (GHS)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Task 31: Line chart (2 marks)

Draw a **line chart of `monthly_sales`** with markers, a title and labelled axes.

In [ ]:
monthly_sales.plot(marker="o")
plt.title("Monthly sales")
plt.xlabel("Month")
plt.ylabel("Sales (GHS)")
plt.tight_layout()
plt.show()

---
## Part I: Saving and reloading (5 marks)
**Focus:** exporting cleaned data and confirming it round-trips correctly.

### Task 32: Save to CSV (2 marks)

Save `df` to a file named **`sales_clean.csv`** **without** the index column.

In [ ]:
df.to_csv("sales_clean.csv", index=False)

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 32", lambda: os.path.exists("sales_clean.csv"), "3cbc87c7")


### Task 33: Reload and verify (3 marks)

Read `sales_clean.csv` back into a new DataFrame named **`df_check`**, making sure `order_date` is read as a **date** (not text). Display its shape.

In [ ]:
df_check = pd.read_csv("sales_clean.csv", parse_dates=["order_date"])
df_check.shape

In [ ]:
# Self-check (do not edit): run this cell after completing the task
check("Task 33", lambda: (df_check.shape, pd.api.types.is_datetime64_any_dtype(df_check["order_date"])), "2e2d5975")


---
## Final reflection (not marked)
Double-click and answer honestly. This is how you find what to practise next.

1. Which **two tasks** were hardest, and what did you do to solve them?
2. Which pandas skill do you feel **most confident** about now?
3. Which skill do you want to **practise more**?
4. Name **one business question** about this sales data you would investigate next, and how you would approach it.

## Skills checklist
Tick the skills you can now do without looking anything up (change `[ ]` to `[x]`):

- [ ] Convert a dictionary to a DataFrame and inspect it
- [ ] Find and handle missing values and duplicates
- [ ] Fix data types, including dates
- [ ] Create new columns from calculations and conditions
- [ ] Select columns and filter rows with one or many conditions
- [ ] Sort, rank and find top values
- [ ] Summarise with `groupby`, aggregation and pivot tables
- [ ] Analyse data over time
- [ ] Merge two tables
- [ ] Draw and label basic charts
- [ ] Save and reload cleaned data

## Submission
Run **Kernel > Restart & Run All**, make sure no cell shows an error, then save and submit the `.ipynb` file as instructed by your instructor.